# Supervised Fine-Tuning (SFT)

**What you will build**: A complete SFT pipeline -- from raw instruction data to a fine-tuned model that follows instructions. You will implement everything from scratch first, then compare with the TRL library.

**Pedagogy**: Code first, theory second. Each section starts with a working implementation, then asks *why* it works.

---

## 0 | Self-Quiz (Active Recall)

**Answer these from memory before you read any further.** Write your answers in the empty cell below.

1. What is SFT and how does it differ from pretraining?
2. What data format does SFT typically use? Give an example.
3. What loss function is used for SFT? Is it the same as pretraining?
4. Why do we mask the instruction/prompt tokens in the loss computation?
5. What is the typical learning rate for SFT relative to pretraining?
6. Where does SFT sit in the RLHF pipeline (pretraining -> ??? -> reward model -> PPO)?

*Your answers here (double-click to edit):*

1. 
2. 
3. 
4. 
5. 
6. 

---
## 1 | Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets trl accelerate peft

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import json
import copy
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from datasets import load_dataset

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## 2 | SFT Intuition

### What is SFT?

**Supervised Fine-Tuning** is the process of taking a pretrained language model (which only knows how to predict the next token from internet text) and teaching it to **follow instructions** by training on (instruction, response) pairs.

Think of it as:
- **Pretraining** = learning to speak (grammar, facts, reasoning patterns)
- **SFT** = learning to be a helpful assistant (follow instructions, answer questions, match a desired format)

### The key insight

SFT uses the **exact same loss function** as pretraining (next-token prediction / cross-entropy), but on a **curated dataset** of instruction-response pairs. The magic is in the data, not the algorithm.

### Data formats

Let's look at the three most common instruction formats side by side.

In [ ]:
# --- Three common SFT data formats ---

# 1. Alpaca format (Stanford Alpaca)
alpaca_example = {
    "instruction": "Explain the concept of gradient descent in simple terms.",
    "input": "",
    "output": "Gradient descent is like finding the lowest point in a valley while blindfolded. "
              "You feel the slope under your feet and take a step downhill. You repeat this "
              "process, and eventually you reach the bottom. In machine learning, the 'valley' "
              "is the loss function, and the 'steps' are updates to the model's parameters."
}

# 2. ChatML format (used by many modern models)
chatml_example = """<|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user
Explain the concept of gradient descent in simple terms.<|im_end|>
<|im_start|>assistant
Gradient descent is like finding the lowest point in a valley while blindfolded. You feel the slope under your feet and take a step downhill. You repeat this process, and eventually you reach the bottom.<|im_end|>"""

# 3. Llama chat format
llama_chat_example = """<s>[INST] <<SYS>>
You are a helpful AI assistant.
<</SYS>>

Explain the concept of gradient descent in simple terms. [/INST] Gradient descent is like finding the lowest point in a valley while blindfolded. You feel the slope under your feet and take a step downhill. You repeat this process, and eventually you reach the bottom. </s>"""

print("=" * 60)
print("ALPACA FORMAT:")
print("=" * 60)
print(json.dumps(alpaca_example, indent=2))
print()
print("=" * 60)
print("CHATML FORMAT:")
print("=" * 60)
print(chatml_example)
print()
print("=" * 60)
print("LLAMA CHAT FORMAT:")
print("=" * 60)
print(llama_chat_example)

**Key point**: The format itself matters less than **consistency**. The model learns to recognize the format tokens as signals for when to start generating a response. The critical thing is that we only compute loss on the **response** tokens, not the instruction tokens.

---
## 3 | Data Preparation

We will load the Alpaca dataset and prepare it for SFT training. The key step is **label masking**: we set the loss labels for instruction tokens to -100 (PyTorch's ignore_index), so the model is only trained to predict response tokens.

In [ ]:
# --- Load the Alpaca dataset ---
dataset = load_dataset("tatsu-lab/alpaca", split="train")
print(f"Dataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"\nFirst example:")
print(json.dumps(dataset[0], indent=2))

In [ ]:
# --- Load GPT-2 tokenizer ---
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# GPT-2 doesn't have a pad token by default; set it to eos
tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"EOS token: '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"PAD token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

In [ ]:
def format_alpaca(example: dict) -> str:
    """Convert an Alpaca example to a prompt-response string."""
    if example["input"].strip():
        prompt = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n"
        )
    else:
        prompt = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Response:\n"
        )
    return prompt, example["output"]


# Demo
prompt, response = format_alpaca(dataset[0])
print("PROMPT:")
print(prompt)
print("RESPONSE:")
print(response)

In [ ]:
class SFTDataset(Dataset):
    """
    Dataset for supervised fine-tuning with proper label masking.

    The key idea: we tokenize the full sequence (prompt + response),
    but set labels to -100 for all prompt tokens so the loss is only
    computed on the response.
    """

    def __init__(self, dataset, tokenizer, max_length=256, max_samples=None):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []

        n = min(len(dataset), max_samples) if max_samples else len(dataset)

        for i in range(n):
            prompt_text, response_text = format_alpaca(dataset[i])
            full_text = prompt_text + response_text + tokenizer.eos_token

            # Tokenize prompt and full text separately to find the boundary
            prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
            full_ids = tokenizer.encode(full_text, add_special_tokens=False)

            # Truncate if necessary
            if len(full_ids) > max_length:
                full_ids = full_ids[:max_length]

            prompt_len = min(len(prompt_ids), len(full_ids))

            # Create labels: -100 for prompt tokens, actual token ids for response
            labels = [-100] * prompt_len + full_ids[prompt_len:]

            # Pad to max_length
            pad_len = max_length - len(full_ids)
            input_ids = full_ids + [tokenizer.pad_token_id] * pad_len
            labels = labels + [-100] * pad_len  # don't compute loss on padding
            attention_mask = [1] * len(full_ids) + [0] * pad_len

            self.examples.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

In [ ]:
# --- Create dataset with 1000 examples for speed ---
sft_dataset = SFTDataset(dataset, tokenizer, max_length=256, max_samples=1000)
print(f"SFT dataset size: {len(sft_dataset)}")

# Inspect one example
ex = sft_dataset[0]
print(f"\ninput_ids shape:     {ex['input_ids'].shape}")
print(f"labels shape:        {ex['labels'].shape}")
print(f"attention_mask shape: {ex['attention_mask'].shape}")

# Show the masking in action
tokens = tokenizer.convert_ids_to_tokens(ex['input_ids'][:50])
labels = ex['labels'][:50].tolist()
print("\nFirst 50 tokens with labels (-100 = masked, i.e., instruction):")
print("-" * 70)
for i, (tok, lab) in enumerate(zip(tokens, labels)):
    status = "MASKED (instruction)" if lab == -100 else f"label={lab}"
    print(f"  {i:3d}: {tok:15s} -> {status}")
    if i > 0 and lab != -100 and labels[i-1] == -100:
        print("  >>> BOUNDARY: loss starts being computed here <<<")

### Why mask the instruction?

If we compute loss on the instruction tokens, the model wastes capacity learning to predict the instruction itself (which we provide at inference time anyway). By masking instruction tokens:

1. **All gradient signal goes toward learning good responses** -- the model focuses on the task we actually care about.
2. **Prevents the model from memorizing instruction patterns** -- we want generalization to new instructions.
3. **Aligns training with inference** -- at inference, we provide the instruction and ask the model to generate only the response.

This is a simple but critical detail. Getting the label masking wrong is a common source of bugs in SFT pipelines.

**Insider Tip:** At frontier labs, SFT data quality is considered MORE important than quantity. Anthropic and OpenAI employ teams of expert human annotators (PhDs, domain specialists) who write high-quality demonstrations. The difference between good and great SFT comes down to data, not training tricks. The LIMA paper (Zhou et al. 2023) demonstrated that just 1,000 carefully curated examples can outperform models trained on 52K lower-quality examples. In interviews, emphasize that you understand data curation is the bottleneck, not algorithmic sophistication.

---
## 4 | SFT Training Loop from Scratch

We will implement the full training loop manually. No training frameworks -- just PyTorch, so you understand every detail.

In [ ]:
# --- Load base GPT-2 model ---
base_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
base_model.config.pad_token_id = tokenizer.pad_token_id

# Save a copy for before/after comparison
base_model_copy = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
base_model_copy.config.pad_token_id = tokenizer.pad_token_id
base_model_copy.eval()

n_params = sum(p.numel() for p in base_model.parameters())
print(f"GPT-2 parameters: {n_params:,}")

In [ ]:
# --- Manual SFT training loop ---
train_loader = DataLoader(sft_dataset, batch_size=4, shuffle=True)

optimizer = torch.optim.AdamW(base_model.parameters(), lr=5e-5, weight_decay=0.01)

# Linear warmup scheduler
num_epochs = 3
total_steps = num_epochs * len(train_loader)
warmup_steps = total_steps // 10

def get_lr(step):
    if step < warmup_steps:
        return step / warmup_steps
    return max(0.0, 1.0 - (step - warmup_steps) / (total_steps - warmup_steps))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

print(f"Training for {num_epochs} epochs, {len(train_loader)} steps/epoch")
print(f"Total steps: {total_steps}, Warmup steps: {warmup_steps}")

In [ ]:
# --- The actual training loop ---
train_losses = []
step = 0

base_model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    n_batches = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Forward pass
        outputs = base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,  # HuggingFace handles the cross-entropy + label masking
        )
        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping (important for stability)
        torch.nn.utils.clip_grad_norm_(base_model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        n_batches += 1
        step += 1
        train_losses.append(loss.item())

        if step % 50 == 0:
            lr = scheduler.get_last_lr()[0]
            print(f"  Step {step:4d}: loss = {loss.item():.4f}, lr = {lr:.2e}")

    avg_loss = epoch_loss / n_batches
    print(f"Epoch {epoch + 1}/{num_epochs} -- avg loss: {avg_loss:.4f}")
    print()

In [ ]:
# --- Plot training loss ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw loss per step
axes[0].plot(train_losses, alpha=0.5, linewidth=0.5)
# Smoothed loss (rolling average)
window = 20
if len(train_losses) >= window:
    smoothed = np.convolve(train_losses, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(train_losses)), smoothed, color='red', linewidth=2, label=f'{window}-step avg')
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("SFT Training Loss")
axes[0].legend()
axes[0].grid(True)

# Learning rate schedule
lr_values = [get_lr(s) * 5e-5 for s in range(total_steps)]
axes[1].plot(lr_values)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("Learning Rate Schedule (Linear Warmup + Decay)")
axes[1].grid(True)

plt.tight_layout()
plt.show()

### Understanding the loss computation in detail

Let's break down exactly what HuggingFace's `labels` argument does internally, so you can explain it in an interview.

In [ ]:
# --- Manual loss computation (equivalent to what HF does internally) ---
base_model.eval()
example = sft_dataset[0]
input_ids = example["input_ids"].unsqueeze(0).to(device)
labels = example["labels"].unsqueeze(0).to(device)

with torch.no_grad():
    outputs = base_model(input_ids=input_ids)
    logits = outputs.logits  # (1, seq_len, vocab_size)

# Shift: logits[t] predicts token[t+1]
shift_logits = logits[:, :-1, :].contiguous()   # (1, seq_len-1, vocab_size)
shift_labels = labels[:, 1:].contiguous()        # (1, seq_len-1)

# Cross-entropy with ignore_index=-100 (masks out instruction + padding)
loss_fct = nn.CrossEntropyLoss(ignore_index=-100, reduction='none')
per_token_loss = loss_fct(
    shift_logits.view(-1, shift_logits.size(-1)),
    shift_labels.view(-1)
)

# Reshape and visualize
per_token_loss = per_token_loss.view(shift_labels.shape)

# Show which tokens contribute to loss
token_strs = tokenizer.convert_ids_to_tokens(input_ids[0][1:])
loss_vals = per_token_loss[0].cpu().numpy()
label_vals = shift_labels[0].cpu().numpy()

print("Token-level loss (0.0 = masked out):")
print("-" * 50)
for i in range(min(40, len(token_strs))):
    if label_vals[i] == -100:
        print(f"  {token_strs[i]:15s} -> MASKED")
    else:
        print(f"  {token_strs[i]:15s} -> loss = {loss_vals[i]:.4f}")

# Total loss (average over non-masked tokens only)
non_masked = per_token_loss[shift_labels != -100]
manual_loss = non_masked.mean()
print(f"\nManual loss: {manual_loss.item():.4f}")

---
## 5 | SFT with TRL Library

Now let's see how the `trl` library (by HuggingFace) abstracts this process. The `SFTTrainer` handles tokenization, label masking, and training in just a few lines.

In [ ]:
from trl import SFTTrainer, SFTConfig

# Load a fresh GPT-2 for the TRL comparison
trl_model = GPT2LMHeadModel.from_pretrained("gpt2")
trl_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
trl_tokenizer.pad_token = trl_tokenizer.eos_token
trl_model.config.pad_token_id = trl_tokenizer.pad_token_id

In [ ]:
# --- Prepare the dataset as text strings ---
# TRL's SFTTrainer can work with a text column directly
def format_for_trl(example):
    prompt, response = format_alpaca(example)
    return {"text": prompt + response + trl_tokenizer.eos_token}

trl_dataset = dataset.select(range(1000)).map(format_for_trl)
print(f"TRL dataset size: {len(trl_dataset)}")
print(f"Sample text (first 200 chars): {trl_dataset[0]['text'][:200]}...")

In [ ]:
# --- SFT with TRL SFTTrainer ---
sft_config = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    max_seq_length=256,  # NOTE: renamed to `max_length` in TRL >= 0.13
    logging_steps=50,
    save_strategy="no",
    gradient_accumulation_steps=1,
    warmup_ratio=0.1,
    weight_decay=0.01,
    report_to="none",
)

trainer = SFTTrainer(
    model=trl_model,
    train_dataset=trl_dataset,
    args=sft_config,
    processing_class=trl_tokenizer,
)

print("Starting TRL SFTTrainer...")
trainer.train()
print("Done!")

### Manual vs. TRL: What does the library abstract away?

| Aspect | Manual Loop | SFTTrainer |
|--------|------------|------------|
| Tokenization | You handle it | Automatic |
| Label masking | You compute the prompt boundary | Can be automatic with `DataCollatorForCompletionOnlyLM` |
| Gradient accumulation | You implement it | Built-in via config |
| Mixed precision (fp16/bf16) | You use `torch.cuda.amp` | Built-in via config |
| Distributed training | You use `DistributedDataParallel` | Built-in via `accelerate` |
| Logging/checkpointing | You implement it | Built-in |
| Learning rate schedule | You implement it | Built-in (cosine, linear, etc.) |

**Interview insight**: Know both levels of abstraction. Be able to write the manual loop (shows depth) and know the library (shows you ship code in practice).

**Insider Tip:** LoRA/QLoRA (Hu et al. 2021, Dettmers et al. 2023) is standard for efficient fine-tuning. Full fine-tuning is used for production runs but LoRA is used for rapid experimentation. Mention this trade-off in interviews. At frontier labs, LoRA is also critical for serving multiple fine-tuned variants from a single base model -- you only swap the small adapter weights. Recent work on DoRA (Weight-Decomposed Low-Rank Adaptation, Liu et al. 2024) and rsLoRA improve on the original LoRA with better training dynamics.

---
## 6 | Evaluation: Before vs. After SFT

Let's compare the base GPT-2 model (before SFT) with our fine-tuned model on a few instruction-following tasks.

In [ ]:
def generate_response(model, tokenizer, instruction, max_new_tokens=100):
    """Generate a response for a given instruction using Alpaca format."""
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )

    generated = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)
    return generated.strip()

In [ ]:
# --- Compare before/after on several instructions ---
test_instructions = [
    "Write a short poem about machine learning.",
    "Explain what a neural network is in one sentence.",
    "List three benefits of exercise.",
    "Translate 'Hello, how are you?' to French.",
]

print("=" * 70)
print("BEFORE vs AFTER SFT")
print("=" * 70)

for instruction in test_instructions:
    print(f"\nINSTRUCTION: {instruction}")
    print("-" * 50)

    before = generate_response(base_model_copy, tokenizer, instruction)
    after = generate_response(base_model, tokenizer, instruction)

    print(f"BEFORE SFT: {before[:200]}")
    print(f"AFTER SFT:  {after[:200]}")
    print()

In [ ]:
# --- Simple quantitative evaluation: response length and format adherence ---
import re

def basic_metrics(model, tokenizer, instructions, n=50):
    """Compute basic metrics on a set of instructions."""
    lengths = []
    non_empty = 0
    contains_answer = 0

    for inst in instructions[:n]:
        response = generate_response(model, tokenizer, inst, max_new_tokens=80)
        lengths.append(len(response.split()))
        if len(response.strip()) > 5:
            non_empty += 1
        # Simple heuristic: does it look like it's trying to answer?
        if not response.startswith("###") and len(response.strip()) > 10:
            contains_answer += 1

    return {
        "avg_length": np.mean(lengths),
        "non_empty_rate": non_empty / len(instructions[:n]),
        "answer_rate": contains_answer / len(instructions[:n]),
    }

# Evaluate on a sample of unseen instructions
eval_instructions = [dataset[i]["instruction"] for i in range(1000, 1020)]

print("Evaluating base model (before SFT)...")
metrics_before = basic_metrics(base_model_copy, tokenizer, eval_instructions, n=20)
print(f"  Avg response length: {metrics_before['avg_length']:.1f} words")
print(f"  Non-empty rate:      {metrics_before['non_empty_rate']:.1%}")
print(f"  Answer rate:         {metrics_before['answer_rate']:.1%}")

print("\nEvaluating SFT model (after training)...")
metrics_after = basic_metrics(base_model, tokenizer, eval_instructions, n=20)
print(f"  Avg response length: {metrics_after['avg_length']:.1f} words")
print(f"  Non-empty rate:      {metrics_after['non_empty_rate']:.1%}")
print(f"  Answer rate:         {metrics_after['answer_rate']:.1%}")

---
## 7 | "Why Does This Work?" / What Can Go Wrong?

### Elaborative interrogation prompts

Answer each question in 2-3 sentences. These are the kind of questions that come up in frontier lab interviews.

#### Why does it work?

1. **Why does training on just ~1000 instruction-response pairs noticeably change GPT-2's behavior?** The model already "knows" language from pretraining -- SFT is just teaching it a new *interface* for accessing that knowledge.

2. **Why is SFT sometimes called "unlocking" the model's capabilities?** The pretrained model already has the ability to answer questions (it saw Q&A on the internet). SFT teaches it to reliably produce that behavior when given the right format cues.

3. **Why does the Alpaca format (### Instruction: / ### Response:) work at all?** The model has seen similar patterns in internet text (markdown headers, Q&A forums). The format tokens become "mode switches" that put the model into response-generation mode.

#### What can go wrong?

4. **Catastrophic forgetting / loss of pretraining knowledge**: If you train too long or with too high a learning rate, the model may lose its general language abilities. This is why SFT uses low learning rates (1e-5 to 5e-5) and few epochs (1-3).

5. **Distributional shift**: The model is trained on one distribution of instructions but deployed on a different distribution. If training data is mostly "write a poem" but users ask "debug this code," performance degrades.

6. **Style mimicry without understanding**: SFT can teach a model to *sound* helpful without actually *being* helpful. The model learns the format and tone of a good response but may produce plausible-sounding nonsense. This is why RLHF/DPO is needed on top of SFT.

7. **Data quality bottleneck**: SFT is only as good as its training data. Low-quality or incorrect responses in the training set directly become model behaviors.

#### "What would break if..." prompts

8. **...you forgot to mask the instruction in the labels?** The model would try to memorize instructions, wasting capacity. It might also learn to generate instructions instead of responses.

9. **...you used a learning rate of 1e-3 instead of 5e-5?** The model would likely diverge or catastrophically forget pretraining knowledge. Large learning rates destroy the pretrained representations in the early layers.

10. **...you trained for 100 epochs instead of 3?** Heavy overfitting to the training set. The model would memorize exact responses rather than learning the general skill of instruction following.

---
## Interview Question Bank: Supervised Fine-Tuning

*These questions are asked at Anthropic, OpenAI, DeepMind, and Meta for senior/principal ML roles. SFT is often the first topic in an RLHF deep-dive interview.*

---

### Question 1: "How would you construct a high-quality SFT dataset?"

**What we're testing:** Data-centric ML thinking. The best candidates know that data quality matters more than model architecture for SFT.

**Good answer:** Describes the core workflow: collect instruction-response pairs, filter for quality (length, formatting, factual accuracy), deduplicate, split into train/eval. Mentions that SFT datasets are surprisingly small compared to pretraining data -- hundreds of thousands, not billions.

**Great answer (Principal-level):** Goes deep on the annotation pipeline: designing detailed annotation guidelines with rubrics, measuring inter-annotator agreement (Cohen's kappa > 0.7 is the industry threshold), handling annotator disagreement through adjudication or majority vote, building curriculum (easy -> hard examples), running data decontamination against benchmarks, and discussing the diversity-quality trade-off (diverse prompts + high-quality responses > lots of similar examples). Mentions that frontier labs like Anthropic invest heavily in specialized annotator teams with domain expertise (math, coding, science).

**Red flag:** Thinks more data is always better. Doesn't consider data quality metrics. Has no opinion on annotation guidelines or quality control.

**Follow-up:** "How do you handle disagreement between annotators?" (Tests understanding of annotation quality: adjudication by senior annotators, measuring systematic vs random disagreement, using disagreement as a signal for ambiguous instructions that should be revised or removed.)

---

### Question 2: "What are the failure modes of SFT?"

**What we're testing:** Understanding of what can go wrong -- a signal of practical experience.

**Good answer:** Overfitting to the small SFT dataset (especially with a large model), catastrophic forgetting of pretraining knowledge, and sensitivity to hyperparameters (learning rate too high destroys capabilities, too low wastes compute).

**Great answer (Principal-level):** All of the above, plus: distribution shift between SFT data and deployment (the model encounters prompts very different from training), reward hacking via style mimicry (the model learns to produce responses that *look* helpful -- correct formatting, confident tone -- without actually being correct), evaluation metrics that don't capture true capability (BLEU/ROUGE are useless for SFT evaluation, but many teams still use them). Also discusses: the alignment tax (SFT can reduce raw capability on benchmarks while improving usability), and how SFT can create sycophantic behavior by training on data where annotators always agree with the user.

**Red flag:** Only mentions overfitting. Doesn't know about catastrophic forgetting. Can't reason about distribution shift.

**Follow-up:** "You've SFT'd a model and it performs great on your eval set but users complain it's unhelpful. What do you investigate?" (Tests debugging skills: eval set may not be representative, style mimicry, sycophancy, or the model may be refusing too aggressively.)

---

### Question 3: "LoRA vs full fine-tuning -- when to use which?"

**What we're testing:** Practical engineering judgment about efficiency vs quality trade-offs.

**Good answer:** LoRA (Low-Rank Adaptation) freezes the base model and trains low-rank update matrices, reducing trainable parameters by 100-1000x. Use LoRA for rapid experimentation, limited compute, or when you need multiple task-specific adaptors. Use full fine-tuning when you have the compute budget and want maximum quality.

**Great answer (Principal-level):** Discusses the quality gap: full fine-tuning consistently outperforms LoRA by a measurable margin on complex tasks, but LoRA closes the gap as rank increases. Mentions DoRA (Weight-Decomposed Low-Rank Adaptation) as a method that bridges the gap. Knows that LoRA rank 64-128 is typical for 70B models, and that LoRA is essential for serving multiple fine-tuned variants efficiently (can swap LoRA adaptors at inference time without loading separate models). Discusses QLoRA for fine-tuning on consumer GPUs. Knows that at frontier labs, full fine-tuning is used for the final production model, while LoRA is used for experimentation and ablation studies.

**Red flag:** Claims LoRA is always as good as full fine-tuning. Doesn't know what rank to use. Can't explain why LoRA works (the low-rank hypothesis for weight updates).

**Follow-up:** "How would you decide the LoRA rank for a specific task?" (Tests systematic thinking: start with rank 16, measure quality, increase to 32/64 if needed, compare against full FT on a held-out set.)

---
## Production Implementation Notes: SFT at Frontier Scale

*What SFT looks like when you're training a model that will serve millions of users.*

### The Textbook vs. Reality Gap

| Component | Textbook Version (this notebook) | Production Version (frontier labs) |
|-----------|--------------------------------|-----------------------------------|
| **Dataset size** | 1K examples on GPT-2 | 100K-1M expert-curated examples on 70B+ models |
| **Data format** | Simple instruction-response | Multi-turn conversations, system prompts, tool use, structured outputs |
| **Training** | Single GPU, few epochs | 64-256 GPUs with FSDP, 2-4 epochs with careful LR schedule |
| **Loss masking** | Basic instruction masking | Per-role masking in multi-turn, special token handling, tool call masking |
| **Evaluation** | Loss on held-out set | Human eval, model-as-judge, capability benchmarks, safety red-teaming |

### Scale Numbers You Should Know

- **SFT dataset size:** Frontier labs use 100K-1M high-quality examples. NOT millions -- quality beats quantity. (Constitutional AI, for reference, fine-tunes on model-self-revised responses in its SL-CAI stage before RLAIF; exact data counts vary by paper stage.)
- **Annotation cost:** $10-50 per example for expert-level quality (math, coding, scientific reasoning). Simple instruction-following is cheaper ($2-5). Total data cost for one SFT round: $500K-$5M.
- **Training time for SFT on 70B:** 4-12 hours on 64-256 H100 GPUs. Much shorter than pretraining (months).
- **Learning rate:** Typically 1e-5 to 5e-5 for full fine-tuning (10x lower than pretraining). This is one of the most sensitive hyperparameters.
- **Epochs:** 2-4 epochs. Beyond that, overfitting risk increases sharply given the small dataset size.
- **Batch size:** Effective batch size of 64-256 (with gradient accumulation). Larger batches stabilize training but require more memory.

### Engineering Challenges Not in Papers

1. **Multi-turn formatting:** Real SFT data is multi-turn conversation. Getting the chat template right (special tokens, role markers, masking) is surprisingly error-prone. A single token offset in masking can waste an entire training run.
2. **Data versioning and lineage:** SFT datasets evolve constantly as annotation guidelines change. Tracking which data version produced which model is a critical MLOps challenge.
3. **Decontamination:** You must ensure your SFT data doesn't contain benchmark answers. This requires n-gram matching against all major benchmarks (MMLU, GSM8K, HumanEval, etc.).
4. **Training stability:** Even SFT can have loss spikes on certain data batches. These often correspond to extremely long or unusually formatted examples. Robust data preprocessing and gradient clipping are essential.
5. **Evaluation is the hard part:** Training is straightforward; knowing when to stop and which checkpoint to ship is where most of the difficulty lies. Teams typically evaluate every N steps on a diverse eval suite and use model-as-judge for qualitative assessment.

### What Monitoring Looks Like in Production

- **Training loss curve:** Should decrease smoothly. Spikes indicate data issues.
- **Gradient norm:** Should be stable. Large norms suggest problematic examples.
- **Per-category eval:** SFT models are evaluated separately on coding, math, creative writing, instruction following, safety, etc.
- **A/B testing:** Before deploying a new SFT model, it's tested against the current production model via human preference evaluation (typically 500-2000 comparison pairs).

---
## How This Gets Tested in Interviews

### Where SFT Questions Appear

| Company | Round | Format | Depth |
|---------|-------|--------|-------|
| **Anthropic** | Onsite (research depth) | Discussion + design | Deep -- expects data pipeline design, quality metrics, failure mode analysis |
| **OpenAI** | Phone screen + onsite | Discussion | Moderate -- focus on practical SFT experience |
| **DeepMind** | Onsite (ML systems) | System design | Moderate -- SFT as part of larger RLHF pipeline design |
| **Meta (GenAI)** | Onsite | Coding + discussion | Moderate -- implement training loop, discuss data strategies |
| **Cohere / AI21** | Technical screen | Discussion + coding | SFT is central to their product, expect deep practical knowledge |

### Time Expectations

- **"Design an SFT data pipeline"**: 20-30 minute discussion. You should be able to whiteboard the full pipeline: data collection -> annotation -> quality control -> decontamination -> formatting -> training -> evaluation.
- **"Implement SFT training loop"**: 15-20 minutes coding. Should include proper loss masking, gradient accumulation, and basic logging.
- **"Debug this SFT run"**: 10-15 minute problem-solving. Given symptoms (loss curve, sample outputs), diagnose the issue.

### Senior vs. Principal Expectations

**senior ML engineer:**
- Implement an SFT training loop correctly
- Understand instruction masking and why it matters
- Know LoRA basics and when to use it
- Describe a reasonable data collection strategy
- List common failure modes

**principal ML engineer:**
- All of the above, plus:
- Design the full annotation pipeline: guidelines, annotator selection, calibration, quality metrics
- Quantify trade-offs: how many examples are needed? What quality threshold? What's the ROI of more data vs better data?
- Discuss curriculum design: ordering examples from simple to complex
- Know the interaction between SFT and downstream RLHF: how SFT quality affects RM training and PPO stability
- Have an opinion on: when is SFT sufficient vs when do you need RLHF? (hint: SFT teaches format, RLHF teaches quality)
- Discuss evaluation methodology: why automated metrics fail, how to design human eval protocols

### Preparation Checklist

- [ ] Implement SFT with proper instruction masking from scratch (time yourself)
- [ ] Be ready to discuss: "What makes a good SFT example?" (have 3-4 concrete quality criteria)
- [ ] Know the LoRA paper: what is the rank, where are adaptors placed, what is the initialization?
- [ ] Have a concrete answer for: "How do you evaluate an SFT model beyond loss?" (human eval, model-as-judge, task-specific benchmarks)
- [ ] Understand the relationship: SFT teaches the model *what format* to respond in; RLHF teaches it *what quality* to aim for

---
## 8 | Flashcard Summary (Anki Export Format)

| # | Question | Answer |
|---|----------|--------|
| 1 | What is Supervised Fine-Tuning (SFT)? | Training a pretrained LM on (instruction, response) pairs to teach it to follow instructions. Uses the same cross-entropy loss as pretraining but on curated data. |
| 2 | How does SFT differ from pretraining? | Same loss function, but: (1) curated instruction-response data instead of raw internet text, (2) much lower learning rate (typically 1e-5 to 5e-5, vs pretraining peak LRs of ~1.5e-4 to 3e-4), (3) few epochs (1-3), (4) loss computed only on response tokens. |
| 3 | What is label masking in SFT and why is it important? | Setting loss labels to -100 for instruction/prompt tokens so the model is only trained to predict response tokens. Prevents wasting capacity on memorizing prompts. |
| 4 | What are the three common SFT data formats? | (1) Alpaca format: instruction/input/output fields, (2) ChatML: `<\|im_start\|>role` tags, (3) Llama chat: `[INST]...[/INST]` tags |
| 5 | What is catastrophic forgetting in the context of SFT? | The model loses its pretrained knowledge (language understanding, world knowledge) when fine-tuned too aggressively. Mitigated by low learning rate, few epochs, and sometimes weight regularization. |
| 6 | Where does SFT sit in the RLHF pipeline? | Pretraining -> SFT -> Reward Model Training -> RL (PPO). Note: DPO is not RL -- it replaces the RM + PPO stages by optimizing directly on preference data rather than following RM training. SFT creates the initial instruction-following model that preference optimization further refines. |
| 7 | What is the typical learning rate for SFT? | 1e-5 to 5e-5, roughly 10-30x below typical pretraining peak LRs (~1.5e-4 to 3e-4). This preserves pretrained representations while adapting the model's behavior. |
| 8 | What is "style mimicry" in SFT? | The model learns the *format* and *tone* of helpful responses (e.g., bullet points, polite language) without necessarily improving the *accuracy* of its answers. |
| 9 | Why is data quality the main bottleneck for SFT? | Unlike pretraining (which can use raw internet text), SFT requires high-quality instruction-response pairs. Errors in training data become model behaviors directly. |
| 10 | What happens if you forget to mask instruction tokens? | The model wastes gradient signal on predicting instruction text, learns to generate instructions instead of just responses, and trains less efficiently. |

---
## 9 | Paper Reading Guide

### "Training language models to follow instructions with human feedback" -- Ouyang et al. (2022)

**Link**: [https://arxiv.org/abs/2203.02155](https://arxiv.org/abs/2203.02155)

### 3-Sentence Summary

This paper (commonly called "InstructGPT") describes OpenAI's three-step process for aligning language models: (1) supervised fine-tuning on human demonstrations, (2) training a reward model on human comparisons, and (3) optimizing the SFT model against the reward model using PPO. The key finding is that a 1.3B parameter InstructGPT model was preferred by human evaluators over the 175B parameter GPT-3, demonstrating that alignment training is more important than model scale. The paper also introduces important concepts like the "alignment tax" (slight regression on NLP benchmarks after RLHF) and the role of data quality in SFT.

### Reading Guide: Key Sections

| Section | What to focus on | Time |
|---------|-----------------|------|
| Abstract + Section 1 (Intro) | The three-step pipeline diagram (Figure 2). Understand the flow: SFT -> RM -> PPO. | 10 min |
| Section 3.1 (SFT Data) | How was the SFT data collected? What instructions did labelers write? What was the data size (~13k)? | 10 min |
| Section 3.2 (SFT Training) | Training details: 16 epochs, cosine LR schedule, residual dropout. Why 16 epochs (overfitting was beneficial)? | 10 min |
| Section 3.3-3.4 (RM + RL) | Reward model architecture and PPO training. Focus on the objective function (Eq. 2). | 15 min |
| Section 4 (Results) | Figure 3: InstructGPT 1.3B vs GPT-3 175B. Tables 1-3: human evaluation results. | 10 min |
| Section 5 (Discussion) | Alignment tax, limitations, the cost of the methodology. | 10 min |

### Interview-Relevant Questions from This Paper

1. Walk me through the InstructGPT training pipeline from pretraining to deployment.
2. Why did they train SFT for 16 epochs when that clearly overfits on NLP benchmarks? (Answer: human preference continued improving even past overfitting.)
3. How was the reward model structured? What loss function did it use?
4. What is the "alignment tax" and how significant is it?
5. How would you collect SFT data if you were starting a new alignment effort today?

### Related Papers to Read Next

- **Alpaca** (Taori et al. 2023): 52K SFT examples generated with OpenAI's text-davinci-003 (GPT-3.5 era) via Self-Instruct -- shows you can distill instruction following from a stronger model
- **LIMA** (Zhou et al. 2023): 1000 carefully curated examples are enough for strong SFT -- emphasizes data quality over quantity
- **DPO** (Rafailov et al. 2023): Direct Preference Optimization -- replaces PPO step with simpler supervised learning
- **Llama 2** (Touvron et al. 2023): Detailed SFT + RLHF recipe at scale, excellent technical writing
- **Llama 3** (Meta 2024): Updated training recipe with more emphasis on data quality and post-training
- **Tulu 3** (Lambert et al., 2024): Systematic study of SFT and post-training recipes, strong open reproduction
- **LoRA** (Hu et al. 2021): Low-rank adaptation for parameter-efficient fine-tuning -- essential for practical SFT
- **QLoRA** (Dettmers et al. 2023): 4-bit quantized LoRA enabling fine-tuning of 65B models on a single 48GB GPU